# 🎤 ChuckleNet — WavLM Audio Laughter Detection

**Colab GPU | Real Audio from GDrive | May 2026**

Trains `microsoft/wavlm-base-plus` (frozen encoder + MLP head) on real stand-up comedy audio.

**Data:** 71 stand-up comedy videos, 15K aligned utterances, 3.2 GB audio
**Checkpoints:** `phaseA_best.pt`, `gated_fusion.pt`, `wavlm_optimized_v2.pt`

## 1. Setup & Mount GDrive

In [ ]:
# @title 1. Install + Mount + Imports
!pip install -q transformers librosa soundfile accelerate gdown matplotlib

import torch, numpy as np, json, time, os, subprocess, librosa
from collections import Counter
from functools import lru_cache
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
BASE = '/content/drive/MyDrive'
AUDIO_DIR = f'{BASE}/chuckle_checkpoints/audio'
SEGMENTS_PATH = f'{BASE}/chuckle_checkpoints/aligned_utterances.jsonl'
OUTPUT_DIR = f'{BASE}/chuckle_checkpoints'

for p in [AUDIO_DIR, SEGMENTS_PATH]:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  {status}: {p}")

if os.path.exists(AUDIO_DIR):
    n = len([f for f in os.listdir(AUDIO_DIR) if f.endswith('.mp3')])
    print(f"  Audio files: {n}")

## 2. Load Aligned Utterances

In [ ]:
# @title 2. Load segments with train/val split
SR = 16000
MAX_SAMPLES = 15000

print(f"Loading from {SEGMENTS_PATH}...")
samples = []

with open(SEGMENTS_PATH) as f:
    for i, line in enumerate(f):
        if i >= MAX_SAMPLES:
            break
        d = json.loads(line)
        vid = d.get('video_id', '')
        audio_path = f'{AUDIO_DIR}/{vid}.mp3'
        
        if not os.path.exists(audio_path):
            continue
        
        label = d.get('label_any', 0)
        start = d.get('start')
        end = d.get('end')
        
        if start is not None and end is not None and end > start:
            samples.append({
                'audio_path': audio_path,
                'start': start,
                'end': end,
                'label': int(label),
                'video_id': vid
            })

# Split by video (80/20)
import random; random.seed(42)
vids = list(set(s['video_id'] for s in samples))
random.shuffle(vids)
split = int(0.8 * len(vids))
train_vids = set(vids[:split])

train = [s for s in samples if s['video_id'] in train_vids]
val = [s for s in samples if s['video_id'] not in train_vids]

lc = Counter(s['label'] for s in train)
vc = Counter(s['label'] for s in val)
print(f"Train: {len(train)} (laughter={lc[1]}, {100*lc[1]/len(train):.1f}%)")
print(f"Val:   {len(val)} (laughter={vc[1]}, {100*vc[1]/len(val):.1f}%)")

# @title 3. Audio extraction (low-memory segment load)

# Load only the segment needed, not full MP3 files.
# Colab T4 RAM is limited; streaming avoids OOM.

def extract_segment(audio_path, start, end, pad_ms=50):
    try:
        offset = max(0, start - pad_ms/1000 - 0.5)
        duration = (end - start) + pad_ms/1000 * 2 + 1.0
        y, _ = librosa.load(audio_path, sr=SR, mono=True,
                            offset=offset, duration=duration)
    except:
        return np.zeros(int(0.3 * SR), dtype=np.float32)
    if len(y) < int(0.1 * SR):
        y = np.pad(y, (0, int(0.1 * SR) - len(y)))
    return y.astype(np.float32)

# Test
s = train[0]
test = extract_segment(s['audio_path'], s['start'], s['end'])
print(f"Test extract: shape={test.shape}, dur={len(test)/SR:.2f}s")

In [ ]:
# @title 3. Audio extraction (low-memory segment load)

# Load only the segment needed, not full MP3 files.
# Colab T4 RAM is limited; streaming avoids OOM.

def extract_segment(audio_path, start, end, pad_ms=50):
    try:
        offset = max(0, start - pad_ms/1000 - 0.5)
        duration = (end - start) + pad_ms/1000 * 2 + 1.0
        y, _ = librosa.load(audio_path, sr=SR, mono=True,
                            offset=offset, duration=duration)
    except:
        return np.zeros(int(0.3 * SR), dtype=np.float32)
    if len(y) < int(0.1 * SR):
        y = np.pad(y, (0, int(0.1 * SR) - len(y)))
    return y.astype(np.float32)

# Test
s = train[0]
test = extract_segment(s['audio_path'], s['start'], s['end'])
print(f"Test extract: shape={test.shape}, dur={len(test)/SR:.2f}s")

## 4. Load WavLM-Base+

In [ ]:
# @title 4. Load frozen WavLM encoder
print("Loading microsoft/wavlm-base-plus...")
t0 = time.time()

encoder = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
encoder.to(device)
encoder.eval()

total = sum(p.numel() for p in encoder.parameters())
print(f"Loaded in {time.time()-t0:.1f}s | {total/1e6:.1f}M params")

## 5. Model: WavLM + MLP Classifier

In [ ]:
# @title 5. Build classifier

class WavLMLaughterClassifier(torch.nn.Module):
    def __init__(self, enc):
        super().__init__()
        self.encoder = enc
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 256), torch.nn.ReLU(), torch.nn.Dropout(0.25),
            torch.nn.Linear(256, 64),  torch.nn.ReLU(), torch.nn.Dropout(0.25),
            torch.nn.Linear(64, 2)
        )
    
    def forward(self, waveforms):
        with torch.no_grad():
            wav_list = [w.cpu().numpy() for w in waveforms]
            inputs = feature_extractor(wav_list, sampling_rate=SR, return_tensors="pt", padding=True)
            inputs = {k: v.to(waveforms.device) for k, v in inputs.items()}
            hidden = self.encoder(**inputs).last_hidden_state.mean(dim=1)
        return self.classifier(hidden)

model = WavLMLaughterClassifier(encoder).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} params ({100*trainable/total:.1f}% of total)")

## 6. DataLoader & Training Loop

In [ ]:
# @title 6. Dataset + DataLoader (num_workers=0 to avoid OOM)

class UtteranceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        audio = extract_segment(s['audio_path'], s['start'], s['end'])
        return torch.from_numpy(audio), torch.tensor(s['label'], dtype=torch.long)

BATCH = 16  # Smaller batch for T4 GPU RAM
train_loader = DataLoader(UtteranceDataset(train), batch_size=BATCH,
                          shuffle=True, num_workers=0)
val_loader = DataLoader(UtteranceDataset(val), batch_size=BATCH,
                        shuffle=False, num_workers=0)
print(f"Batches: {len(train_loader)} train / {len(val_loader)} val")

## 7. Train

In [ ]:
# @title 7. Training loop

EPOCHS = 5
LR = 1e-3
POS_WEIGHT = 5.0

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, POS_WEIGHT]).to(device))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_f1 = 0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Train
    model.train()
    tp, tl, tloss = [], [], 0
    for bi, (audio, y) in enumerate(train_loader):
        audio, y = audio.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(audio), y)
        loss.backward()
        optimizer.step()
        tloss += loss.item()
        tp.extend(model(audio).argmax(1).cpu().tolist())
        tl.extend(y.cpu().tolist())
        if (bi+1) % 50 == 0:
            print(f"  Batch {bi+1}/{len(train_loader)} loss={loss.item():.4f}")
    
    scheduler.step()
    train_f1 = f1_score(tl, tp, zero_division=0)
    
    # Validate
    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for audio, y in val_loader:
            audio, y = audio.to(device), y.to(device)
            vp.extend(model(audio).argmax(1).cpu().tolist())
            vl.extend(y.cpu().tolist())
    
    val_f1 = f1_score(vl, vp, zero_division=0)
    val_p = precision_score(vl, vp, zero_division=0)
    val_r = recall_score(vl, vp, zero_division=0)
    
    elapsed = time.time() - t0
    print(f"Epoch {epoch}/{EPOCHS} ({elapsed:.0f}s) | Train F1={train_f1:.4f} | Val F1={val_f1:.4f} P={val_p:.4f} R={val_r:.4f}")
    history.append({'epoch': epoch, 'train_f1': train_f1, 'val_f1': val_f1, 'val_p': val_p, 'val_r': val_r, 'time': elapsed})
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'val_f1': val_f1, 'history': history}, f'{OUTPUT_DIR}/wavlm_phaseA_best.pt')
        print(f"  -> Saved best (F1={val_f1:.4f})")

print(f"\nBest Val F1: {best_val_f1:.4f}")

## 8. Results & Save

In [ ]:
# @title 8. Plot + Save

# Training curve
epochs_l = [h['epoch'] for h in history]
plt.figure(figsize=(10, 5))
plt.plot(epochs_l, [h['train_f1'] for h in history], 'b-o', label='Train F1')
plt.plot(epochs_l, [h['val_f1'] for h in history], 'g-o', label='Val F1')
plt.axhline(y=0.55, color='r', linestyle='--', label='Target F1=0.55')
plt.xlabel('Epoch'); plt.ylabel('F1 Score')
plt.title('ChuckleNet - WavLM Laughter Detection (Real Audio)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/training_curve.png', dpi=100, bbox_inches='tight')
plt.show()

# Save final
torch.save(model.state_dict(), f'{OUTPUT_DIR}/wavlm_phaseA_final.pt')

# Summary
print("="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"Best Val F1:  {best_val_f1:.4f}")
print(f"Samples:      {len(train)} train / {len(val)} val")
print(f"Model:        WavLM-Base+ (frozen) + MLP")
print(f"Saved:        {OUTPUT_DIR}/wavlm_phaseA_final.pt")
print(f"Best:         {OUTPUT_DIR}/wavlm_phaseA_best.pt")
for h in history:
    print(f"  E{h['epoch']}: train={h['train_f1']:.4f} val={h['val_f1']:.4f} ({h['time']:.0f}s)")